# Exclusion Summary & Customer Count per Month
**Data Sources:**
- `EC_DATA` ao 20260430
- `pal_pol` (ao 20260514)
- `Daily Active Manpower` (ao 20260514)

This notebook replicates the Exclusion Summary waterfall and Customer Count per Month using the `inf_pol` table.

In [ ]:
import pandas as pd
import numpy as np
from datetime import date

pd.set_option('display.float_format', '{:,.0f}'.format)
pd.set_option('display.max_columns', None)

## 1. Load Data

In [ ]:
# ── Load inf_pol ──────────────────────────────────────────────────────────────
# Replace the path below with the actual file location
inf_pol = pd.read_csv('inf_pol.csv')   # or pd.read_parquet / pd.read_excel

# Parse dates
inf_pol['EFFECTIVE_DT'] = pd.to_datetime(inf_pol['EFFECTIVE_DT'], errors='coerce')

print(f'inf_pol shape: {inf_pol.shape}')
inf_pol.head(3)

## 2. Exclusion Summary — Waterfall

Each step records **Unique Customers** remaining after applying the exclusion.

In [ ]:
summary_rows = []

def snap(label, df, id_col='OWNER_ALPHA_ID'):
    """Append a row to the exclusion summary."""
    unique_customers = df[id_col].nunique()
    unique_policies   = df['POLNUM'].nunique() if 'POLNUM' in df.columns else np.nan
    summary_rows.append({
        'Step': label,
        'Unique Customers': unique_customers,
        'Unique Policies' : unique_policies
    })
    return df

In [ ]:
# ── Step 1 : EC_DATA inf_pol — all records ────────────────────────────────────
df = inf_pol.copy()
snap('EC_DATA inf_pol — Unique Customers (all)', df)

In [ ]:
# ── Step 2 : Keep Policy Status 20, 26, 32 only ───────────────────────────────
# POLICY_STATUS codes: 20 = In-Force, 26 = Paid-Up, 32 = (site-specific)
VALID_STATUS = [20, 26, 32]

# Adjust dtype if POLICY_STATUS is stored as string
if df['POLICY_STATUS'].dtype == object:
    VALID_STATUS = [str(s) for s in VALID_STATUS]

df = df[df['POLICY_STATUS'].isin(VALID_STATUS)]
snap('Customers with 20-26 and 32 Policy Status', df)

In [ ]:
# ── Step 3 : With ALPHA_ID (non-null, non-blank) ──────────────────────────────
df = df[df['OWNER_ALPHA_ID'].notna() & (df['OWNER_ALPHA_ID'].astype(str).str.strip() != '')]
snap('With ALPHA_ID', df)

In [ ]:
# ── Step 4 : Individual Customers only (exclude company-owned) ────────────────
# Assumes inf_pol has COMPANY_OWN_IND or similar; adjust column name if needed
if 'COMPANY_OWN_IND' in df.columns:
    df = df[df['COMPANY_OWN_IND'] != 1]
else:
    print('Column COMPANY_OWN_IND not found — skipping company exclusion.')
snap('Individual Customers', df)

In [ ]:
# ── Step 5 : With Gender ──────────────────────────────────────────────────────
# OWNER_GENDER expected from inf_pol or joined from INF_Customer
gender_col = 'OWNER_GENDER' if 'OWNER_GENDER' in df.columns else None

if gender_col:
    df = df[df[gender_col].notna() & (df[gender_col].astype(str).str.strip() != '')]
else:
    print('Gender column not found in inf_pol. Join INF_Customer first if needed.')
snap('With Gender', df)

In [ ]:
# ── Step 6 : With DOB ─────────────────────────────────────────────────────────
dob_col = 'OWNER_DOB' if 'OWNER_DOB' in df.columns else None

if dob_col:
    df[dob_col] = pd.to_datetime(df[dob_col], errors='coerce')
    df = df[df[dob_col].notna()]
else:
    print('DOB column not found in inf_pol. Join INF_Customer first if needed.')
snap('With DOB', df)

In [ ]:
# ── Step 7 : AGE >= 18 as of December 31, 2026 ───────────────────────────────
REFERENCE_DATE = pd.Timestamp('2026-12-31')

if dob_col and dob_col in df.columns:
    df['AGE_AS_OF_20261231'] = (REFERENCE_DATE - df[dob_col]).dt.days // 365
    df = df[df['AGE_AS_OF_20261231'] >= 18]
else:
    print('Skipping age filter — DOB column unavailable.')
snap('AGE => 18 as of December 31, 2026', df)

In [ ]:
# ── Step 8 : Final Base (deduplicated to unique customers) ────────────────────
final_base = df.drop_duplicates(subset='OWNER_ALPHA_ID')
snap('Final Base', final_base, id_col='OWNER_ALPHA_ID')

In [ ]:
# ── Print Exclusion Summary ───────────────────────────────────────────────────
exclusion_summary = pd.DataFrame(summary_rows)
exclusion_summary['Drop from Previous'] = (
    exclusion_summary['Unique Customers']
    .diff()
    .fillna(0)
    .astype(int)
)

print('=' * 70)
print('EXCLUSION SUMMARY')
print('=' * 70)
print(exclusion_summary.to_string(index=False))

## 3. Customer Count per Month (by Owner DOB Month)

Using the **Final Base** — count unique customers by birth month.

In [ ]:
MONTH_MAP = {
    1: 'January', 2: 'February', 3: 'March',    4: 'April',
    5: 'May',     6: 'June',     7: 'July',      8: 'August',
    9: 'September', 10: 'October', 11: 'November', 12: 'December'
}

if dob_col and dob_col in final_base.columns:
    final_base = final_base.copy()
    final_base['OWNER_DOB_MONTH']      = final_base[dob_col].dt.month
    final_base['OWNER_DOB_MONTH_NAME'] = final_base['OWNER_DOB_MONTH'].map(MONTH_MAP)

    count_by_month = (
        final_base
        .groupby(['OWNER_DOB_MONTH', 'OWNER_DOB_MONTH_NAME'], sort=True)
        ['OWNER_ALPHA_ID'].nunique()
        .reset_index()
        .rename(columns={'OWNER_ALPHA_ID': 'Unique Customers'})
    )

    count_by_month['% Share'] = (
        count_by_month['Unique Customers'] /
        count_by_month['Unique Customers'].sum() * 100
    ).round(2)

    print('=' * 50)
    print('CUSTOMER COUNT PER MONTH (Owner DOB Month)')
    print('=' * 50)
    print(count_by_month.to_string(index=False))
    print(f"\nTOTAL: {count_by_month['Unique Customers'].sum():,}")
else:
    print('DOB column unavailable — cannot compute Customer Count per Month.')

In [ ]:
# ── Optional: Bar chart of count by DOB month ─────────────────────────────────
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(
        count_by_month['OWNER_DOB_MONTH_NAME'],
        count_by_month['Unique Customers'],
        color='steelblue', edgecolor='white'
    )
    ax.set_title('Customer Count by Owner DOB Month — Final Base', fontsize=14, fontweight='bold')
    ax.set_xlabel('Birth Month')
    ax.set_ylabel('Unique Customers')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
except NameError:
    print('count_by_month not available — check prior cell.')

## 4. Export Final Base

In [ ]:
final_base.to_csv('final_base_inf_pol.csv', index=False)
print(f'Exported final_base_inf_pol.csv  —  {len(final_base):,} rows')